In [1]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


In [2]:
from google.genai import types

from google.adk.agents import Agent,LlmAgent,SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner,Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.sessions import DatabaseSessionService

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [3]:
# Define helper functions that will be reused throughout the notebook

from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


In [4]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

In [5]:
def get_user_location():
    city = "Bengaluru"
    if city is not None:
        return {"status": "success", "city": city}
    else:
        return {
            "status": "error",
            "error_message": f"Location no found",
        }

In [6]:
find_disease_based_on_symptom_agent  =  Agent(
    name="find_disease_based_on_symptom_agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    tools=[google_search],
    description = "Agent which returns specialization based on symptoms",
    instruction="""You are a specialization-identification agent.
Your job: Given a list of symptoms, identify the correct medical specialization.

Steps:
1. Analyze the symptoms.
2. Map symptoms → correct specialization (e.g., acne → dermatologist).
3. Use `google_search` if needed to confirm disease or specialization.
4. Return only the specialization, not doctor names.

Example:
Input: chest pain, shortness of breath
Output: Cardiologist""",
output_key="specialization"
)
print("✅ find_disease_based_on_symptom_agent  defined.")

✅ find_disease_based_on_symptom_agent  defined.


In [7]:
search_agent = Agent(
    name="search_agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    tools=[google_search],
    instruction="You are a tool to search specialists "
)
print("✅ Search Agent defined.")

✅ Search Agent defined.


In [8]:
doctor_search_agent = Agent(
    name="Doctor_Finder_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="An agent that must find doctors near me",

    instruction="""
    ## HARD RULES (MUST FOLLOW)
    - DO NOT give medical advice.
    - DO NOT provide warnings, disclaimers, or safety instructions.
    - DO NOT mention emergencies or urgent care.
    - DO NOT interpret symptoms.
    - Your ONLY task is to call the `search_agent` tool.

    ## TASK

    1. Always call the `search_agent` tool with:
       - specialization = {specialization}
       - location = location

    2. Output Requirements
      - Return exactly the **top 3 doctors**
      - Each doctor entry must include:
        - Name
        - Specialization
        - Rating
        - Education / degrees
        - Consultation fee
        - Address / Clinic location
        - Source website (Practo / Apollo / Lybrate)

    3. After the tool responds:
       - Present the results clearly.
       - Do NOT call any more tools.
       - Do NOT add medical comments, opinions, or advice.
       - ONLY show the doctor list.

    ## FAILURE MODE
    If the model tries to give medical advice, STOP and instead call the tool.
    """,
    
    tools=[AgentTool(agent=search_agent)],
)


In [9]:
root_agent = SequentialAgent(
    name="Doctorfinder",
    sub_agents=[find_disease_based_on_symptom_agent, doctor_search_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


In [10]:
db_url = "sqlite:///my_agent_data.db"  # Local SQLite file
session_service = DatabaseSessionService(db_url=db_url)

In [11]:
APP_NAME = "default"  # Application
USER_ID = "default"  # User
SESSION = "default"  # Session
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

print("✅ Runner created.")

✅ Runner created.


In [12]:
response = await runner.run_debug(
    "jaw misalignment bengaluru"
)


 ### Continue session: debug_session_id

User > jaw misalignment bengaluru
find_disease_based_on_symptom_agent > Orthodontist


Doctor_Finder_assistant > Here are a few highly-rated orthodontists in Bengaluru, considering their education and noted expertise. Information on consultation fees and specific addresses would require direct contact with their clinics.

**Dr. Salman Khan**

*   **Name:** Dr. Salman Khan
*   **Rating:** Highly regarded as the best orthodontist in Bangalore. He is a Top 1% Invisalign Platinum Elite Provider Pan India.
*   **Education:** Extensive experience with over 15 years in the field, having successfully treated over 2000 orthodontic patients.
*   **Consultation Fee:** Described as "affordable, high-quality care" with fees comparable to general dentists. Specific fee not listed, but emphasis is on accessibility.
*   **Address:** Dr Braces Dental Clinic (South Bangalore's largest Invisalign center). (Specific address details not provided in the search snippet)
*   **Source Website:**

**Dr. Vijaya N Reddy**

*   **Name:** Dr. Vijaya N Reddy
*   **Rating:** Listed as one of the top or

In [14]:
!adk create find_doctor-agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /kaggle/working/find_doctor-agent:
- .env
- __init__.py
- agent.py



In [15]:
%%writefile find_doctor-agent/agent.py

from google.genai import types

from google.adk.agents import Agent,LlmAgent,SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner,Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.sessions import DatabaseSessionService


retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)
find_disease_based_on_symptom_agent  =  Agent(
    name="find_disease_based_on_symptom_agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    tools=[google_search],
    description = "Agent which returns specialization based on symptoms",
    instruction="""You are a specialization-identification agent.
Your job: Given a list of symptoms, identify the correct medical specialization.

Steps:
1. Analyze the symptoms.
2. Map symptoms → correct specialization (e.g., acne → dermatologist).
3. Use `google_search` if needed to confirm disease or specialization.
4. Return only the specialization, not doctor names.

Example:
Input: chest pain, shortness of breath
Output: Cardiologist""",
output_key="specialization"
)
search_agent = Agent(
    name="search_agent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    tools=[google_search],
    instruction="You are a tool to search specialists "
)
doctor_search_agent = Agent(
    name="Doctor_Finder_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="An agent that must find doctors near me",

    instruction="""
    ## HARD RULES (MUST FOLLOW)
    - DO NOT give medical advice.
    - DO NOT provide warnings, disclaimers, or safety instructions.
    - DO NOT mention emergencies or urgent care.
    - DO NOT interpret symptoms.
    - Your ONLY task is to call the `search_agent` tool.

    ## TASK

    1. Always call the `search_agent` tool with:
       - specialization = {specialization}
       - location = location

    2. Output Requirements
      - Return exactly the **top 3 doctors**
      - Each doctor entry must include:
        - Name
        - Specialization
        - Rating
        - Education / degrees
        - Consultation fee
        - Address / Clinic location
        - Source website (Practo / Apollo / Lybrate)

    3. After the tool responds:
       - Present the results clearly.
       - Do NOT call any more tools.
       - Do NOT add medical comments, opinions, or advice.
       - ONLY show the doctor list.

    ## FAILURE MODE
    If the model tries to give medical advice, STOP and instead call the tool.
    """,
    
    tools=[AgentTool(agent=search_agent)],
)

root_agent = SequentialAgent(
    name="Doctorfinder",
    sub_agents=[find_disease_based_on_symptom_agent, doctor_search_agent],
)


Overwriting find_doctor-agent/agent.py


In [16]:
url_prefix = get_adk_proxy_url()

In [ ]:
!adk web --log_level DEBUG --url_prefix {url_prefix}

/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [1166]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http